## Переменные из докера

In [ ]:
import os

In [ ]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

## Получение данных из бд

In [ ]:
!pip install psycopg2-binary

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine(DATABASE_URL)

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

In [ ]:
df_activity_types = pd.read_sql("SELECT * FROM activity_types;", engine)
df_activity_types

In [ ]:
df_fitness_data = pd.read_sql("SELECT * FROM fitness_data;", engine)
df_fitness_data

In [ ]:
df = pd.merge(
    df_fitness_data,
    df_activity_types,
    left_on="activity_type_id",
    right_on="id",
)
df

In [ ]:
df["timestamp"] = pd.to_datetime(df["recorded_at"], utc=True)
df = df.sort_values("timestamp")
df

## Графики

In [ ]:
!pip install plotly
!pip install seaborn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

pio.renderers.default = "notebook"

### Передвижения по карте (от нейронки, не получалось сделать самому)

In [ ]:
fig = px.line_map(
    df,
    lat="lat",
    lon="lon",
    zoom=12,
    height=700,
    title="Траектория передвижения",
    hover_data={"timestamp": True},
)

fig.update_layout(map_style="open-street-map")


fig.add_trace(
    go.Scattermap(
        lat=df["lat"],
        lon=df["lon"],
        mode="markers",
        marker=dict(size=8, color="lightblue"),
        hovertext=df["timestamp"].dt.strftime("%Y-%m-%d %H:%M"),
        hoverinfo="text",
        name="Промежуточные точки",
        showlegend=False,
    )
)

start = df.iloc[0]
fig.add_trace(
    go.Scattermap(
        lat=[start["lat"]],
        lon=[start["lon"]],
        mode="markers+text",
        marker=dict(size=20, color="green", symbol="circle"),
        text=f"Старт<br>{start['timestamp'].strftime('%Y-%m-%d %H:%M')}",
        textposition="top right",
        name="Старт",
        hoverinfo="text",
        showlegend=True,
    )
)

end = df.iloc[-1]
fig.add_trace(
    go.Scattermap(
        lat=[end["lat"]],
        lon=[end["lon"]],
        mode="markers+text",
        marker=dict(size=20, color="red", symbol="circle"),
        text=f"Финиш<br>{end['timestamp'].strftime('%Y-%m-%d %H:%M')}",
        textposition="top right",
        name="Финиш",
        hoverinfo="text",
        showlegend=True,
    )
)

fig.update_traces(line=dict(width=4, color="blue"))

fig.show()

### Количество километров по дням и часам

In [ ]:
df["day_of_week"] = df["timestamp"].dt.weekday
df["hour"] = df["timestamp"].dt.hour

heatmap_data = (
    df.pivot_table(
        values="distance_km",
        index="day_of_week",
        columns="hour",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(index=range(7), columns=range(24), fill_value=0)
    .values
)

In [ ]:
# нормализация, но как будто без нее лучше
# heatmap_data = heatmap_data / heatmap_data.max()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(
    heatmap_data,
    cmap="viridis",
    annot=False,
    yticklabels=["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"],
)
plt.yticks(rotation=0)
plt.xlabel("Час дня")
plt.ylabel("День недели")
plt.title("Тепловая карта пройденных киллометров по дням и часам")
plt.show()

### Сумма пройденных шагов с течением времени

In [ ]:
df["cumulative_steps"] = df["steps"].cumsum()

plt.figure(figsize=(12, 6))

plt.plot(df["timestamp"], df["cumulative_steps"])

plt.title("Сумма пройденных шагов с течением времени")
plt.xlabel("Дата и время")
plt.ylabel("Количество шагов")
plt.tight_layout()
plt.grid()

plt.show()

## Разные статистики

### Корреляции

In [ ]:
# df[['steps', 'distance_km', 'kilocalories']].corr().style.background_gradient(cmap="viridis")

corr = df[["steps", "distance_km", "kilocalories"]].corr()
np.fill_diagonal(
    corr.values, np.nan
)  # чтобы не отвлекаться на единицы на диагонале
corr.style.background_gradient(axis=None, cmap="viridis")

### Средние по типу активности

In [ ]:
df.groupby("type_name")[["steps", "distance_km", "kilocalories"]].mean()